[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lmassaron/finetuning/blob/main/chapter_04/augmenting.ipynb)

### Listing 4.1: Defining the Augmenter Prompt

In [ ]:
AUGMENTER_SYSTEM_PROMPT = (
    "You are an expert financial data scientist specializing in data augmentation. "
    "Your task is to take a financial news headline and create 2 distinct variations of it. "
    "For each variation:\n"
    "1. Replace PII (names, specific locations, small companies) with realistic but fake or generic equivalents.\n"
    "2. Replace numerical figures with different but plausible values that preserve the original market implication.\n"
    "3. Rephrase the sentence completely while strictly maintaining the original sentiment.\n"
    'Format your output as a JSON list of strings: ["Variation 1", "Variation 2"]'
)


### Listing 4.2: Defining the Judge Prompt

In [ ]:
JUDGE_SYSTEM_PROMPT = (
    "You are a quality control judge for financial data augmentation. "
    "Compare the 'Original' headline with its 'Augmented' version. "
    "Evaluate based on:\n"
    "1. PII Removal: Names/locations/companies are replaced?\n"
    "2. Figure Diversification: Numbers are different but same implication?\n"
    "3. Rephrasing: Sentence structure is different?\n"
    "4. Sentiment Preservation: Original sentiment is strictly maintained?\n\n"
    "Provide a Score (0-10) and a brief justification. "
    "End your response with 'Final Score: X/10' where X is the score."
)

### Listing 4.3: Prompt building functions

In [ ]:
def build_augment_prompt(sentence: str, sentiment: str) -> str:
    return (
        f'Original Headline: "{sentence}"\n'
        f"Sentiment: {sentiment}\n\n"
        f"Generate 2 augmented variations in JSON format."
    )

def build_judge_prompt(original: str, augmented: str, sentiment: str) -> str:
    return (
        f"Original: {original}\n"
        f"Augmented: {augmented}\n"
        f"Expected Sentiment: {sentiment}\n\n"
        "Evaluate the quality of this augmentation."
    )

### Listing 4.4: Model and Infrastructure Configuration

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
DATASET_ID = "lmassaron/FinancialPhraseBank"
OUTPUT_PATH = "data/FinancialPhraseBank_augmented_judged"
HF_OUTPUT_REPO = "lmassaron/FinancialPhraseBank_augmented_judged"

BATCH_SIZE = 8
MAX_NEW_TOKENS = 512
JUDGE_THRESHOLD = 8                          

### Listing 4.5: Quantization Configuration and function for model loading

In [ ]:
import torch
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import json
from tqdm import tqdm
import re

QUANTIZATION_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

def load_model(model_id: str):
    print(f"Loading {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.float16,
        quantization_config=QUANTIZATION_CONFIG,
        device_map="auto",
    )
    model.eval()
    return tokenizer, model

tokenizer, model = load_model(MODEL_ID)

### Listing 4.6: Functions for Batch Output Response Generation

In [ ]:
def generate_batch(tokenizer, model, chats):
    inputs = tokenizer(
        chats, return_tensors="pt", padding=True, truncation=True, max_length=1024
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)


def generate_responses(tokenizer, model, system_prompt, user_prompts):
    chats = [
        tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": up},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        for up in user_prompts
    ]
    return generate_batch(tokenizer, model, chats)

### Listing 4.7: Locating a JSON List in Generated Text

In [ ]:
def parse_variations(raw_output):
    try:
        start = raw_output.find("[")
        end = raw_output.rfind("]") + 1
        if start != -1 and end != -1:
            return json.loads(raw_output[start:end])
    except json.JSONDecodeError:
        pass
    return []

def parse_judge_score(judgment):
    match = re.search(r"Final Score: (\d+)/10", judgment)
    if match:
        return int(match.group(1))
    return 5  # Default neutral score if parsing fails

### Listing 4.8: Function to Batch Process the Entire Dataset

In [ ]:
LABEL_MAP = {0: "negative", 1: "neutral", 2: "positive"}

In [ ]:
def process_dataset(tokenizer, model, dataset, limit=None):
    results = []
    sentences = dataset["sentence"]
    labels = dataset["label"]
    if limit:
        sentences, labels = sentences[:limit], labels[:limit]

    for start in tqdm(range(0, len(sentences), BATCH_SIZE), desc="Augmenting & Judging"):
        batch_sentences = sentences[start : start + BATCH_SIZE]
        batch_labels = labels[start : start + BATCH_SIZE]
        aug_prompts = [
            build_augment_prompt(s, LABEL_MAP[l])
            for s, l in zip(batch_sentences, batch_labels)
        ]
        aug_raw = generate_responses(tokenizer, model, AUGMENTER_SYSTEM_PROMPT, aug_prompts)

        all_judge_prompts = []
        variation_metadata = [] # Keep track of what variation belongs to what label

        for orig_s, label, raw in zip(batch_sentences, batch_labels, aug_raw):
            variations = parse_variations(raw)
            sentiment = LABEL_MAP[label]

            results.append({
                "sentence": orig_s,
                "sentiment": sentiment,
                "label": label,
                "is_augmented": False,
                "judge_score": 10,
                "judge_comment": "Original data",
            })

            if not variations:
                continue

            for v in variations:
                all_judge_prompts.append(build_judge_prompt(orig_s, v, sentiment))
                variation_metadata.append({"variation": v, "label": label})

        if all_judge_prompts:
            judgments = generate_responses(tokenizer, model, JUDGE_SYSTEM_PROMPT, all_judge_prompts)

            for meta, j in zip(variation_metadata, judgments):
                results.append({
                    "sentence": meta["variation"],
                    "label": meta["label"],
                    "is_augmented": True,
                    "judge_score": parse_judge_score(j),
                    "judge_comment": j,
                })
                
    return results

### Listing 4.9: Loading and Augmenting the Original Dataset

In [ ]:
raw = load_dataset(DATASET_ID)
output_splits = {}

for split_name, data in raw.items():
    print(f"\n── Processing split: {split_name} ──")
    results = process_dataset(tokenizer, model, data)
    filtered_results = [r for r in results if r["judge_score"] >= JUDGE_THRESHOLD]
    output_splits[split_name] = Dataset.from_list(filtered_results)

output_ds = DatasetDict(output_splits)
output_ds.save_to_disk(OUTPUT_PATH)
print(f"\nAugmented and evaluated dataset saved to: {OUTPUT_PATH}")

### Listing 4.10: Reporting the Average Evaluation Score

In [ ]:
df = output_ds["train"].to_pandas()
augmented_df = df[df["is_augmented"]]

print(f"Total augmented examples: {len(augmented_df)}")
print(f"Average Judge Score: {augmented_df['judge_score'].mean():.2f}/10")

if not augmented_df.empty:
    best = augmented_df.sort_values("judge_score", ascending=False).iloc[0]
    print("\n--- Top Quality Augmentation ---")
    print(f"Sentence: {best['sentence']}")
    print(f"Score   : {best['judge_score']}/10")
    print(f"Comment : {best['judge_comment']}")

In [ ]:
ex = output_ds["train"][0]
print(f"Sentence  : {ex['sentence']}")
print(f"Sentiment : {ex['sentiment']}")
print(f"Augmented : {ex['is_augmented']}")
print(f"Explanation: {ex['explanation']}")